In [1]:
import re
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

In [2]:
data_clean = "goodreads_onestar_clean copy.csv"   # cleaned data, one-star reviews only
TEXT_COL = "review_text"   # name of the column with the review text
MIN_WORDS = 10             # skip reviews shorter than this many words
SEED = 42 # random seed for reproducibility

HIDE = ["alice", "wonderland", "carroll", "silverstein", "peter"] # common titles and authors to hide

In [3]:
one_star = pd.read_csv(data_clean) # turning into a DataFrame
one_star = one_star.dropna(subset=[TEXT_COL]) # dropping reviews with no text
one_star = one_star[one_star[TEXT_COL].str.split().str.len() >= MIN_WORDS] # splitting review text into list of words, counting them, and keeping only reviews with at least MIN_WORDS words

print(f"1-star reviews: {len(one_star)}") # printing the number of 1-star reviews after filtering

1-star reviews: 8801


### Flagging reviews about "appropriateness"

In [4]:
def compare_groups(texts_a, texts_b):  # scoring words and phrases that are distinctive to one group vs another
    texts_a = list(texts_a) # a positive score = more typical of group A, negative score = more typical of group B
    texts_b = list(texts_b)

    counter = CountVectorizer( # using CountVectorizer to count words and phrases in the two groups
        ngram_range=(1, 2), # defining upper and lower limit of words to be extracted
        min_df=15,    # words/phrases must appear in 15+ reviews
        token_pattern=r"(?u)\b\w[\w']*\b",   # keeps negations and other words with apostrophes like "don't" 
    )
    counts = counter.fit_transform(texts_a + texts_b) # makes table of counts of each word/phrase in each review
        
    a_review_number = len(texts_a) # how many reviews in group A and group B
    b_review_number = len(texts_b)
    count_a = np.asarray(counts[:a_review_number].sum(axis=0)).ravel() # selects group A reviews, sums the counts of each term across all reviews, and flattens into list of numbers
    count_b = np.asarray(counts[a_review_number:].sum(axis=0)).ravel() # selects group B reviews, sums the counts of each term across all reviews, and flattens into list of numbers

    baseline = count_a + count_b # how many times each term appears in all reviews (group A + group B)
    baseline_total = baseline.sum() # sums across all terms to get the total number of term appearances in all reviews

    odds_a = np.log((count_a + baseline) / (count_a.sum() + baseline_total - count_a - baseline)) # computes the log-odds of each term appearing in group A vs group B, with a baseline to avoid division by zero
    odds_b = np.log((count_b + baseline) / (count_b.sum() + baseline_total - count_b - baseline))
    difference = odds_a - odds_b

    # Turn the difference into a score that accounts for how much data there is
    uncertainty = np.sqrt(1 / (count_a + baseline) + 1 / (count_b + baseline)) # this calculates uncertainty for each term based on its counts in both groups and the baseline 
    score = difference / uncertainty # dividing to get a z-score-like measure

    results = pd.DataFrame({ # putting everything into a DataFrame 
        "term": counter.get_feature_names_out(),
        "score": score,
        "count_A": count_a,
        "count_B": count_b,
    })

        
    for name in HIDE: # hiding common titles and authors from the results
        results = results[~results["term"].str.contains(r"\b" + name + r"\b")]

    return results.sort_values("score", ascending=False).reset_index(drop=True) # sorting the results by score in descending order and resetting the index


In [5]:
# catching reviews containing inappropriate, not appropriate, unsuitable, or offensive, and their variants
TRIGGER_PATTERN = (   # deciding which reviews count as complaints about inappropriateness, using regex to match various phrases
    r"(?:age[- ])?inappropriate\w*" # use of the word "inappropriate" or "age-inappropriate"
    r"|\b(?:not|isn't|wasn't|aren't|weren't)\s+(?:\w+\s+){0,2}(?:age[- ])?appropriate" # negating words followed by "age appropriate" or "appropriate"
    r"|unsuitable|not suitable|offensive|offend|vulgar|crude|perverse|immoral|sinful"  # other words/phrases that indicate inappropriateness
    r"|too (?:scary|violent|graphic|dark|mature|disturbing|sexual|adult|intense|creepy)"
    r"|too (?:old|young|mature|advanced) for"
    r"|not (?:for|meant for|intended for) (?:\w+ ){0,2}(?:kids|children|young)"
    r"|not (?:very )?(?:child|kid|family)[- ]friendly"
)
WINDOW = 10   # how many words on each side of trigger_pattern to look at

In [6]:
has_trigger = one_star[TEXT_COL].str.contains(TRIGGER_PATTERN, case=False, regex=True) # checking which reviews contain the trigger pattern, ignoring case and using regex
matched = one_star[has_trigger]
others = one_star[~has_trigger]
print(f"{len(matched):,} of {len(one_star):,} 1-star reviews contain an " # printing the number and percentage of 1-star reviews that contain an inappropriateness complaint
      f"inappropriateness complaint ({len(matched) / len(one_star):.1%})")

257 of 8,801 1-star reviews contain an inappropriateness complaint (2.9%)


In [7]:
def snippet(text, width=12):
    m = re.search(TRIGGER_PATTERN, text, flags=re.I) # finding the first occurrence of the trigger pattern in the text
    before = " ".join(text[:m.start()].split()[-width:]) # taking the words before the trigger pattern, splitting them into a list, and keeping only the last 12 words
    after = " ".join(text[m.end():].split()[:width]) # taking the words after the trigger pattern, splitting them into a list, and keeping only the first 12 words
    return f"{before} [[{m.group(0)}]] {after}" # returning a string that includes the words before and after the trigger pattern


matched = matched.copy()
matched["snippet"] = matched[TEXT_COL].apply(snippet) # running the function snippet on each review in the matched DataFrame and creating a new column called "snippet" that contains the result

pd.set_option("display.max_colwidth", None) # letting the entire snippet show up in the output, instead of being truncated
print("\n25 random examples (read these first):")
print(matched[["book_id", "snippet"]].sample(min(25, len(matched)), # printing 25 random examples of the matched reviews, showing the book_id and the snippet, and using a random seed for reproducibility
                                             random_state=SEED).to_string())


25 random examples (read these first):
       book_id                                                                                                                                                                     snippet
6403   6054449         cute couple". Not only does the dismissal of true literary art somewhat [[offend]] me, but the description is grossly inaccurate. My three main thoughts throughout
7205  22033266                         up. For instance, she spends a chapter on the doctrine of the [[sinful]] nature of humankind--that people are born with selfish desires and want to
3603     19302          confine itself to morality tales, but this book crosses the line into [[immoral]] ity-without-consequences tales and there is nothing positive to be found within.
873      30118                  children will never hear this trash again. I can't get over how [[not for children]] these poems are and they're very obviously not meant to adults. YUCK.
105     237564           

In [8]:
distinct = compare_groups(matched[TEXT_COL], others[TEXT_COL]) # running the compare_groups function on the matched and others DataFrames to get the most distinctive words in the matched reviews
distinct = distinct[~distinct["term"].str.contains("appropriat|suitab|offen")] # removing the trigger words from the results, so that the most distinctive words are not just the trigger words themselves
distinct.to_csv("words_inappropriate_vs_other_1star.csv", index=False)
print("\nMost distinctive words in reviews that call something inappropriate:")
print(distinct.head(40).round(2).to_string())


Most distinctive words in reviews that call something inappropriate:
             term  score  count_A  count_B
6         old for   4.03       18        5
7         too old   3.32       18       15
8            rose   3.26       23       30
10          lowry   2.96       19       25
11            god   2.89       55      213
12           dark   2.79       45      159
13      political   2.78       18       26
14             oz   2.75       25       54
15  www goodreads   2.71       15       18
16  goodreads com   2.66       15       19
17       the dark   2.43       21       49
18      https www   2.37       14       22
19         people   2.36      145     1135
20         my kid   2.35       17       34
21          bible   2.27       23       66
22      christian   2.21       25       80
23        appeals   2.15       10       13
24       children   2.15      182     1627
25          of oz   2.10       14       29
26          https   2.07       14       30
27          susan   2.05   

#### CSV file containing random sample of 200 reviews about "appropriateness" with column to add HAND LABELS

In [9]:
# taking a random sample of 200 reviews that contain inappropriateness complaints
to_code = matched[["book_id", TEXT_COL, "snippet"]].sample(
    min(200, len(matched)), random_state=SEED)
to_code["reason"] = ""
to_code.to_csv("inappropriate_to_code.csv", index=False) # saving reviews to a CSV file for hand labeling
print("\nSaved inappropriate_to_code.csv with", len(to_code), "reviews to label")


Saved inappropriate_to_code.csv with 200 reviews to label


### Looking at NOT-flagged reviews (reviews not about "appropriateness")

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

print("\nWords most typical of the NOT-flagged reviews:")
print(distinct.sort_values("score").head(40).round(2).to_string(index=False)) # printing the 40 words with the lowest score, which are the most typical of the NOT-flagged reviews



Words most typical of the NOT-flagged reviews:
             term  score  count_A  count_B
                i  -1.67     1172    22292
           boring  -1.16       19      758
               it  -0.99      796    14444
          strange  -0.98        1      221
         couldn't  -0.94       15      555
           a half  -0.93        3      260
            story  -0.92      174     3580
       half stars  -0.92        2      228
             half  -0.91       11      453
             nice  -0.90        8      378
          one and  -0.85        6      309
              put  -0.84       10      401
        it wasn't  -0.84        1      170
           i read  -0.81       25      710
     the pictures  -0.81        3      217
        the first  -0.81       28      768
      interesting  -0.80       16      517
            nancy  -0.80        0      126
           i love  -0.79        5      261
    illustrations  -0.77       40      990
           always  -0.77       15      480
      

In [11]:
print("\n15 random NOT-flagged reviews (first 300 characters):")
for text in others[TEXT_COL].sample(15, random_state=SEED): # a loop taking a random sample of 15 NOT-flagged reviews
    print("-", text[:300].replace("\n", " "), "\n")


15 random NOT-flagged reviews (first 300 characters):
- Might have given this a 2, as it's slight but might appeal to older kids. But the last page foldout means this book's shelf life will be shortened, which makes my investment of library money in it even more depressing. 

- Did not like this; the cute illustrations had sooo much going on that the point of the story was lost. Took me a few pages to even figure it out/find the object as it was so not direct. For the target age group I feel like this would be just a HUGE miss unless you want to find another nose/face/tail 

- This is the worst kind of "literature" to expose your children to. It's filled with stereotypes and fake accents. It's sad that Judy Schachner doesn't know any better. 

- Would have been good except it glorified Zen and a Zen master. That is something that doesn't need to be mentioned in a young kids book. 

- Not really feeling this one. At all. Sometimes I think that in between Yolen getting a really good sto

#### Topic Model of NOT-flagged reviews

In [12]:
NUMBER_OF_TOPICS = 8
tfidf = TfidfVectorizer(stop_words="english", min_df=10, max_df=0.5) # using TF-IDF to assign weights to words in the reviews
X = tfidf.fit_transform(others[TEXT_COL]) # transforming the NOT-flagged reviews into a matrix
nmf = NMF(n_components=NUMBER_OF_TOPICS, random_state=SEED, init="nndsvda", max_iter=400) # using NMF to find topics in the NOT-flagged reviews, with 8 topics, a random seed for reproducibility, and a maximum of 400 iterations
doc_topics = nmf.fit_transform(X) # transforming the matrix of NOT-flagged reviews into a matrix of topic weights for each review
terms = tfidf.get_feature_names_out() # getting the feature names (words) from the TF-IDF vectorizer

others = others.copy()
others["topic"] = doc_topics.argmax(axis=1) # assigning each review to the topic with the highest weight
for i, weights in enumerate(nmf.components_): # a loop that goes through each topic and prints the top 10 words for that topic, along with the number of reviews assigned to that topic
    top_words = [terms[j] for j in weights.argsort()[::-1][:10]]
    n_reviews = (others["topic"] == i).sum()
    print(f"\nTopic {i} ({n_reviews:,} reviews): {', '.join(top_words)}")


Topic 0 (2,481 reviews): children, books, little, good, way, child, time, series, love, reading

Topic 1 (643 reviews): didn, care, finish, enjoy, like, wasn, characters, work, make, appeal

Topic 2 (1,073 reviews): read, books, ve, hated, school, reading, grade, remember, time, worst

Topic 3 (1,175 reviews): story, illustrations, text, cute, nice, pictures, words, good, liked, stars

Topic 4 (957 reviews): just, couldn, finish, maybe, bad, wasn, funny, really, sorry, weird

Topic 5 (434 reviews): boring, finish, plot, thought, long, ugh, finished, good, reading, really

Topic 6 (859 reviews): don, know, kids, think, understand, kid, want, maybe, remember, classic

Topic 7 (922 reviews): like, really, did, felt, wanted, thought, feel, people, didnt, say


In [13]:
from sklearn.preprocessing import MultiLabelBinarizer

df = pd.read_csv("inappropriate_reviews_handlabels.csv")

df["labels"] = df["labels"].fillna("").str.split(";").apply(
    lambda xs: [x.strip() for x in xs if x.strip()]
)
mlb = MultiLabelBinarizer()
y = pd.DataFrame(mlb.fit_transform(df["labels"]), columns=mlb.classes_, index=df.index)
df = df.join(y)

In [23]:
df.head(1)

,review_id,source_row_id,book_id,review_text,is_inappropriate_complaint,labels,matched_trigger,screening_rule,all_triggers,snippet,...,no,non-factual,preachy,racism,religion,scary/dark,sexism,sexuality,violence,witchcraft
0,ee39ef35634d9dbe72d559db4aded36e,11,156970,"I guess I am too old for this book. Even though I did like the fact it had a female protagonist, it still was a little too ""kiddy"" for me. Damn it I should have read it earlier in my life, maybe I would have given it a 5 stars then.",no,[],too old for,age_fit_phrase,too old for,I guess I am [[too old for]] this book. Even though I did like the fact it had a,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
df["labels"]

labels
[]                                  61
[age]                               24
[racism]                            16
[scary/dark]                        12
[bad-influence, age]                11
[scary/dark, age]                   11
[religion]                           5
[age, crude-language]                4
[religion, age]                      3
[scary/dark, age, illustrations]     3
Name: count, dtype: int64

In [ ]:
df = df[["review_id", "book_id", "review_text", "labels", "notes"]]